# Faza 3.E - generacja + trening wariantu E (LoRA per klasa)

Pełny pipeline E:
1. Pull LoRA z HF (po notebooku 06)
2. Skopiuj `variant_D/` dla 5 klas bez LoRA (recykling)
3. Wygeneruj 5 klas z LoRA do `variant_E/`
4. Filtr CLIP+DINOv2 dla nowych 5 klas
5. Combined manifest_kept.csv
6. Push variant_E do HF
7. Trening 12 runów E (4 modele × 3 seedy)
8. Aggregate A/B/D/E + porównanie per-klasa

**Czas:** ~1h generacja + 2.5h trening = **~3.5h Colab**.

**Hipoteza:** LoRA naprawi Ragdoll (D miało −14pp ConvNeXt). E test: czy Ragdoll wraca do baseline lub wyżej?

## 1. Setup

In [ ]:
import os
REPO_URL = 'https://github.com/micwuj/DLICV.git'
REPO_DIR = '/content/dlicv'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull

In [ ]:
!pip install -q -r requirements.txt peft==0.13.2

In [ ]:
from pathlib import Path
PETS = Path('data/raw/oxford-iiit-pet/images')
if not (PETS.exists() and any(PETS.iterdir())):
    !python scripts/download_pets.py

In [ ]:
from huggingface_hub import login, snapshot_download, whoami
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('brak HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('logged in as:', whoami()['name'])

HF_REPO_ID = 'micwuj/dlicv-synth'

## 2. Pull LoRA + variant_D z HF

In [ ]:
BREEDS_LORA = ['Ragdoll', 'Birman', 'Maine_Coon', 'staffordshire_bull_terrier', 'american_bulldog']
BREEDS_NO_LORA = ['Siamese', 'Persian', 'boxer', 'english_cocker_spaniel', 'english_setter']
N_PER_CLASS_E = 300

# LoRA weights
snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type='dataset',
    allow_patterns='loras/*',
    local_dir='data',
)

# variant_D synth (do recyklingu dla 5 klas bez LoRA)
snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type='dataset',
    allow_patterns='variant_D/**',
    local_dir='data/synthetic',
)

for b in BREEDS_LORA:
    p = Path(f'data/loras/{b}.safetensors')
    print(f'  {b}: {p.exists()} ({p.stat().st_size / 1e6:.1f} MB)' if p.exists() else f'  {b}: MISSING')

## 3. Build variant_E/ skeleton: copy variant_D for 5 non-LoRA breeds

In [ ]:
import shutil

VARIANT_D = Path('data/synthetic/variant_D')
VARIANT_E = Path('data/synthetic/variant_E')
VARIANT_E.mkdir(parents=True, exist_ok=True)

for b in BREEDS_NO_LORA:
    src = VARIANT_D / b
    dst = VARIANT_E / b
    if dst.exists() and any(dst.glob('*.png')):
        print(f'{b}: already copied ({sum(1 for _ in dst.glob("*.png"))} files)')
        continue
    dst.mkdir(parents=True, exist_ok=True)
    for f in src.glob(f'{b}_*.png'):
        shutil.copy(f, dst / f.name)
    print(f'{b}: copied {sum(1 for _ in dst.glob("*.png"))} -> {dst}')

## 4. Generacja 5 klas z LoRA do `variant_E/`

Używa istniejącego `scripts/generate_synthetic.py --variant E --lora-dir data/loras`. Per klasa ~30-40 min na T4. Łącznie ~3h ale tylko dla LoRA breeds.

In [ ]:
lora_breeds_arg = ' '.join(BREEDS_LORA)
!python scripts/generate_synthetic.py \
    --variant E \
    --n-per-class {N_PER_CLASS_E} \
    --lora-dir data/loras \
    --breeds {lora_breeds_arg}

## 5. Sanity check - LoRA-gen vs D-gen side by side

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

for breed in BREEDS_LORA:
    d_imgs = sorted((VARIANT_D / breed).glob('*.png'))[:6]
    e_imgs = sorted((VARIANT_E / breed).glob('*.png'))[:6]
    if not e_imgs:
        print(f'skip {breed}: no E images')
        continue
    fig, axes = plt.subplots(2, 6, figsize=(15, 5))
    for c, p in enumerate(d_imgs[:6]):
        axes[0, c].imshow(Image.open(p)); axes[0, c].axis('off')
    for c, p in enumerate(e_imgs[:6]):
        axes[1, c].imshow(Image.open(p)); axes[1, c].axis('off')
    axes[0, 0].set_ylabel('D (simple)', fontsize=10)
    axes[1, 0].set_ylabel('E (LoRA)', fontsize=10)
    fig.suptitle(f'{breed} — D vs E', fontsize=11)
    plt.tight_layout(); plt.show()

## 6. Filtr CLIP+DINOv2 - dla LoRA breeds osobno, dla non-LoRA recykling z manifestu D

In [ ]:
import csv
import json
import torch
from src.synth.filter import FilterConfig, compute_embeddings, nearest_sim
from src.utils.device import device_info, get_device

device = get_device()
print('device:', device_info(device))

SPLIT_PATH = Path('data/splits/pets_10cls_30perclass_seed0.json')
PETS_IMAGES = Path('data/raw/oxford-iiit-pet/images')
with open(SPLIT_PATH) as f:
    split = json.load(f)
class_to_idx = split['meta']['class_to_idx']

fc = FilterConfig()
FLOOR_CLIP = 0.65
FLOOR_DINOV2 = 0.55

kept_rows = []
score_rows = []

for breed in BREEDS_LORA:
    label = class_to_idx[breed]
    real_paths = [PETS_IMAGES / e['image'] for e in split['train'] if e['label'] == label]
    synth_paths = sorted((VARIANT_E / breed).glob(f'{breed}_*.png'))
    print(f'{breed}: {len(synth_paths)} synth (E LoRA) vs {len(real_paths)} real refs')

    real_clip = compute_embeddings(real_paths, 'clip', fc.clip_model_id, device, batch_size=16)
    real_dino = compute_embeddings(real_paths, 'dinov2', fc.dinov2_model_id, device, batch_size=16)
    synth_clip = compute_embeddings(synth_paths, 'clip', fc.clip_model_id, device, batch_size=16)
    synth_dino = compute_embeddings(synth_paths, 'dinov2', fc.dinov2_model_id, device, batch_size=16)

    clip_sim = nearest_sim(synth_clip, real_clip, top_k=fc.top_k).numpy()
    dino_sim = nearest_sim(synth_dino, real_dino, top_k=fc.top_k).numpy()
    mask = (clip_sim >= FLOOR_CLIP) & (dino_sim >= FLOOR_DINOV2)
    print(f'  kept {mask.sum()}/{len(synth_paths)} ({100*mask.sum()/len(synth_paths):.1f}%)')

    for p, c, d, k in zip(synth_paths, clip_sim, dino_sim, mask):
        score_rows.append({'breed': breed, 'path': p.as_posix(), 'clip_sim': float(c), 'dinov2_sim': float(d), 'kept': bool(k)})
        if k:
            kept_rows.append({'breed': breed, 'path': p.as_posix()})

# Non-LoRA breeds: recykluj z manifestu D
d_manifest_path = VARIANT_D / 'manifest_kept.csv'
with open(d_manifest_path) as f:
    d_rows = list(csv.DictReader(f))
for r in d_rows:
    if r['breed'] in BREEDS_NO_LORA:
        # Update path from variant_D/ to variant_E/
        new_path = r['path'].replace('variant_D/', 'variant_E/')
        kept_rows.append({'breed': r['breed'], 'path': new_path})

print(f'\ntotal kept in variant_E manifest: {len(kept_rows)}')

In [ ]:
# Zapisz manifest i filter_scores dla variant_E
manifest_path = VARIANT_E / 'manifest_kept.csv'
with open(manifest_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['breed', 'path'])
    w.writeheader(); w.writerows(kept_rows)

scores_path = VARIANT_E / 'filter_scores_lora.csv'
if score_rows:
    with open(scores_path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['breed', 'path', 'clip_sim', 'dinov2_sim', 'kept'])
        w.writeheader(); w.writerows(score_rows)

from collections import Counter
by_breed = Counter(r['breed'] for r in kept_rows)
print('per-breed kept:')
for b, n in sorted(by_breed.items()):
    print(f'  {b}: {n}')

## 7. Push variant_E na HF

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
for breed in BREEDS_LORA:
    breed_dir = VARIANT_E / breed
    api.upload_folder(
        folder_path=str(breed_dir),
        path_in_repo=f'variant_E/{breed}',
        repo_id=HF_REPO_ID,
        repo_type='dataset',
        commit_message=f'variant E (LoRA): {breed}',
    )
    print(f'pushed variant_E/{breed}')
for fname in ['manifest_kept.csv', 'filter_scores_lora.csv']:
    p = VARIANT_E / fname
    if p.exists():
        api.upload_file(path_or_fileobj=str(p), path_in_repo=f'variant_E/{fname}',
                        repo_id=HF_REPO_ID, repo_type='dataset', commit_message=f'variant_E metadata: {fname}')

## 8. Trening 12 E runów

**WAŻNE:** usuń stare D outputy z poprzednich sesji jeśli były (skip-existing przykryje istniejące E runy).

In [ ]:
!python scripts/run_all.py --pattern 'E_*.yaml' --overrides configs/colab.yaml --skip-existing --dry-run

In [ ]:
!python scripts/run_all.py --pattern 'E_*.yaml' --overrides configs/colab.yaml --skip-existing

## 9. Aggregate + porównanie A/B/D/E

In [ ]:
!python scripts/aggregate_results.py

In [ ]:
import pandas as pd
summary = pd.read_csv('outputs/_aggregate/summary_by_model_variant.csv')
summary

In [ ]:
# Per-class porownanie A vs D vs E (focus na Ragdoll - czy LoRA naprawil?)
perclass = pd.read_csv('outputs/_aggregate/per_class_accuracy.csv')
ragdoll = perclass[perclass['class'] == 'Ragdoll'].pivot(index='model', columns='variant', values='acc_mean')
print('Ragdoll per-model A vs D vs E:')
print(ragdoll)
print()
stafford = perclass[perclass['class'] == 'staffordshire_bull_terrier'].pivot(index='model', columns='variant', values='acc_mean')
print('staffordshire per-model A vs D vs E:')
print(stafford)

In [ ]:
# Push outputs (lekki dump) na HF
import shutil
lite = Path('/tmp/outputs_lite')
if lite.exists(): shutil.rmtree(lite)
for p in Path('outputs').rglob('*'):
    if p.is_file() and p.suffix in {'.json', '.csv', '.png'}:
        target = lite / p.relative_to('outputs')
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(p, target)
api.upload_folder(folder_path=str(lite), path_in_repo='outputs_lite_E', repo_id=HF_REPO_ID, repo_type='dataset', commit_message='outputs after variant E')
print('pushed lite outputs to HF')